# 30_LOGO_Compare: so sánh tcn_v2 và step2 bằng kiểm chứng chéo theo nhóm phiên quay

**Mục đích.** Trả lời tcn_v2 (126 đặc trưng, augmentation hẹp) hay step2 (546 đặc trưng, augmentation rộng) mạnh hơn, mà không đụng tập test (B1, B3, L3).

**Cách làm.** Gộp train và val của manifest_v2 (266 clip, 8 nhóm phiên quay: B2, H1, H2, H3, L1, L2, L4, L5). Mỗi lượt giữ 1 nhóm ra làm tập kiểm tra, train trên các nhóm còn lại. Early stopping dùng 1 nhóm khác làm val trong (nhóm nhỏ nhất có từ 15 clip trở lên, khác nhóm bị giữ ra). Mỗi lượt train 2 họ x 4 seed (0 đến 3), cùng hàm `fit` và cấu hình của 29_Task9_V2.

**Quy tắc chốt TRƯỚC khi chạy (Sang đồng ý 09/11/2026).**
1. Độ đo chính: accuracy theo khung ở góc 0, trung bình 4 seed, mỗi nhóm một điểm.
2. step2 thắng nếu trung bình theo nhóm của (step2 trừ tcn_v2) >= 1 điểm VÀ step2 hơn ở ít nhất 5/8 nhóm.
3. tcn_v2 thắng nếu chiều ngược lại với cùng ngưỡng. Còn lại là hòa, giữ tcn_v2 (đơn giản hơn).
4. Nếu step2 thắng: chạy test MỘT LẦN cho step2 (seed chọn bằng val loss thấp nhất, ngưỡng chọn bằng val theo quy tắc ACC_TARGET của 29). Nếu không, giữ nguyên kết quả test đã có của tcn_v2 s1.
5. Phải ghi trong báo cáo: tcn_v2 s1 đã được chạy test trước khi có quy tắc này nên test không còn nguyên vẹn cho quyết định đó.

Độ đo phụ (không tham gia quy tắc): accuracy ở góc xoay 45 độ, khoảng tin cậy bootstrap, độ lệch giữa các seed. Tập test không được dùng cho tới cell 7.

In [1]:
#cell 1: nạp định nghĩa từ 29_Task9_V2.ipynb (cell 1, 2, 3 và các hàm metrics, rotated, thr_table), không sửa gì. Cell 1 của 29 cũng nạp test lên GPU nhưng notebook này KHÔNG dùng test trước cell 7
import json, math, time
from pathlib import Path
cands = [Path("29_Task9_V2.ipynb"), Path.home() / "Downloads" / "Data_321" / "29_Task9_V2.ipynb", Path.home() / "Downloads" / "29_Task9_V2.ipynb"]
NB29 = next((p for p in cands if p.exists()), None)
assert NB29 is not None, "Không thấy 29_Task9_V2.ipynb; đặt file cùng thư mục notebook này hoặc trong Downloads/Data_321"
code29 = {i: "".join(c["source"]) for i, c in enumerate(json.loads(NB29.read_text(encoding="utf-8"))["cells"], 1) if c["cell_type"] == "code"}
for i in (1, 2, 3):
    exec(compile(code29[i], f"29_cell{i}", "exec"), globals())
exec(compile(code29[7].split("MAGS, REPEAT")[0], "29_cell7_defs", "exec"), globals())      # THRS, metrics, rotated
exec(compile(code29[9].split("tv = thr_table")[0], "29_cell9_defs", "exec"), globals())    # ACC_TARGET, thr_table
print("Đã nạp từ", NB29)

CUDA: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Tmax: 355
train (221, 355, 126) | video: 221 | khung có tay: 28205
val (45, 355, 126) | video: 45 | khung có tay: 4662
test (39, 355, 126) | video: 39 | khung có tay: 4867
tap              train   val  test
nguon                             
Dataset_bonus     66.0   NaN  24.0
Dataset_hand      63.0  18.0   NaN
dataset_labeled   92.0  27.0  15.0
Đã định nghĩa: ['tcn_v2', 'wide45_lownoise', 'step2'] | đầu vào: {'tcn_v2': 126, 'wide45_lownoise': 126, 'step2': 546}
Checkpoint sẽ lưu ở: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_task9_v2
Đã nạp từ 29_Task9_V2.ipynb


In [2]:
#cell 2: gộp train và val thành 266 clip, tạo 8 lượt (mỗi lượt giữ 1 nhóm ra); test không có trong tập này
Xp = torch.cat([data["train"][0], data["val"][0]]); yp = torch.cat([data["train"][1], data["val"][1]]); Mp = torch.cat([data["train"][2], data["val"][2]])
grp = np.concatenate([meta["train"].nhom.values, meta["val"].nhom.values])
GROUPS = sorted(set(grp)); size = {g: int((grp == g).sum()) for g in GROUPS}
assert not set(grp) & set(meta["test"].nhom), "Có nhóm nằm ở cả pool và test"
assert len(grp) == 266 and len(GROUPS) == 8, (len(grp), GROUPS)
def inner_val(g):                                            # nhóm nhỏ nhất có từ 15 clip trở lên, khác nhóm bị giữ ra (xác định, không phụ thuộc kết quả)
    return min((h for h in GROUPS if h != g and size[h] >= 15), key=lambda h: (size[h], h))
FOLDS = {g: inner_val(g) for g in GROUPS}
FAMS, LOGO_CK = ["tcn_v2", "step2"], BASE / "checkpoints_logo"
print("kích thước nhóm:", size)
for g, iv in FOLDS.items():
    print(f"giữ ra {g} ({size[g]} clip) | val trong {iv} ({size[iv]}) | train {len(grp) - size[g] - size[iv]} clip")

kích thước nhóm: {'B2': 66, 'H1': 33, 'H2': 18, 'H3': 30, 'L1': 36, 'L2': 50, 'L4': 6, 'L5': 27}
giữ ra B2 (66 clip) | val trong H2 (18) | train 182 clip
giữ ra H1 (33 clip) | val trong H2 (18) | train 215 clip
giữ ra H2 (18 clip) | val trong L5 (27) | train 221 clip
giữ ra H3 (30 clip) | val trong H2 (18) | train 218 clip
giữ ra L1 (36 clip) | val trong H2 (18) | train 212 clip
giữ ra L2 (50 clip) | val trong H2 (18) | train 198 clip
giữ ra L4 (6 clip) | val trong H2 (18) | train 242 clip
giữ ra L5 (27 clip) | val trong H2 (18) | train 221 clip


In [3]:
#cell 3: train 8 lượt x 2 họ x 4 seed = 64 model (khoảng vài phút đến chục phút); có checkpoint đúng cấu hình thì tự nạp
for g, iv in FOLDS.items():
    tr_i = torch.tensor(np.where((grp != g) & (grp != iv))[0], device=dev); va_i = torch.tensor(np.where(grp == iv)[0], device=dev)
    tr, va = (Xp[tr_i], yp[tr_i], Mp[tr_i]), (Xp[va_i], yp[va_i], Mp[va_i])
    for name in FAMS:
        for s in SEEDS:
            fit(name, s, LOGO_CK / g / name / f"seed{s}" / "TCN.pt", tr, va, f"logo_{g}")
    print("-" * 80)
print("Xong. Checkpoint ở:", LOGO_CK)

logo_B2 tcn_v2 seed 0: tốt nhất epoch 47 | dừng ở epoch 67 | val_loss 0.2162 | 6s
logo_B2 tcn_v2 seed 1: tốt nhất epoch 27 | dừng ở epoch 47 | val_loss 0.2566 | 3s
logo_B2 tcn_v2 seed 2: tốt nhất epoch 31 | dừng ở epoch 51 | val_loss 0.2494 | 3s
logo_B2 tcn_v2 seed 3: tốt nhất epoch 56 | dừng ở epoch 76 | val_loss 0.2506 | 6s
logo_B2 step2 seed 0: tốt nhất epoch 25 | dừng ở epoch 45 | val_loss 0.2460 | 4s
logo_B2 step2 seed 1: tốt nhất epoch 27 | dừng ở epoch 47 | val_loss 0.2186 | 5s
logo_B2 step2 seed 2: tốt nhất epoch 35 | dừng ở epoch 55 | val_loss 0.2387 | 6s
logo_B2 step2 seed 3: tốt nhất epoch 27 | dừng ở epoch 47 | val_loss 0.2168 | 4s
--------------------------------------------------------------------------------
logo_H1 tcn_v2 seed 0: tốt nhất epoch 63 | dừng ở epoch 83 | val_loss 0.2239 | 8s
logo_H1 tcn_v2 seed 1: tốt nhất epoch 27 | dừng ở epoch 47 | val_loss 0.2266 | 4s
logo_H1 tcn_v2 seed 2: tốt nhất epoch 25 | dừng ở epoch 45 | val_loss 0.2363 | 3s
logo_H1 tcn_v2 seed 3

In [4]:
#cell 4: chấm từng model trên nhóm bị giữ ra: accuracy theo khung ở góc 0 và góc xoay 45 (trung bình 5 lần xoay ngẫu nhiên, cùng dữ liệu xoay cho cả 2 họ); lưu số đúng theo từng clip để bootstrap
REPEAT = 5
rows, per_clip = [], {}
for g in GROUPS:
    idx = torch.tensor(np.where(grp == g)[0], device=dev); X, y, M = Xp[idx], yp[idx], Mp[idx]
    gen = torch.Generator(device=dev).manual_seed(0)
    rot = [rotated(X, M, 45, gen) for _ in range(REPEAT)]
    nfr = M.sum(1).float()
    for name in FAMS:
        v = VARIANTS[name]
        for s in SEEDS:
            net = TCN(v["n_in"]).to(dev); net.load_state_dict(torch.load(LOGO_CK / g / name / f"seed{s}" / "TCN.pt", map_location=dev, weights_only=False)["w"]); net.eval()
            def correct(Xin):
                with torch.no_grad(): P = net(v["feat"](Xin, M))
                return ((P.argmax(-1) == y[:, None]) & M).sum(1).float()
            c0 = correct(X); c45 = torch.stack([correct(Xr) for Xr in rot]).mean(0)
            rows.append({"nhom": g, "ho": name, "seed": s, "acc0": (c0.sum() / nfr.sum() * 100).item(), "acc45": (c45.sum() / nfr.sum() * 100).item()})
            per_clip[(g, name, s)] = (c0.cpu().numpy(), c45.cpu().numpy(), nfr.cpu().numpy())
    print("xong", g)
pd.DataFrame(rows).to_csv(LOGO_CK / "logo_acc.csv", index=False)
print("Đã lưu:", LOGO_CK / "logo_acc.csv")

xong B2
xong H1
xong H2
xong H3
xong L1
xong L2
xong L4
xong L5
Đã lưu: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_logo\logo_acc.csv


In [5]:
#cell 5: áp quy tắc đã chốt, bootstrap 95% (lấy mẫu lại nhóm rồi clip trong nhóm, cùng clip cho cả 2 họ), ghi logo_verdict.json
df = pd.DataFrame(rows)
mg = df.groupby(["nhom", "ho"])[["acc0", "acc45"]].mean().unstack("ho")
tab = pd.DataFrame({"số clip": pd.Series(size),
                    "tcn_v2 acc0": mg[("acc0", "tcn_v2")], "step2 acc0": mg[("acc0", "step2")],
                    "tcn_v2 acc45": mg[("acc45", "tcn_v2")], "step2 acc45": mg[("acc45", "step2")]})
tab["chênh acc0"] = tab["step2 acc0"] - tab["tcn_v2 acc0"]; tab["chênh acc45"] = tab["step2 acc45"] - tab["tcn_v2 acc45"]
print("Accuracy theo khung (%), trung bình 4 seed, nhóm bị giữ ra (chênh = step2 trừ tcn_v2):"); display(tab.round(1))

D = tab["chênh acc0"]; mean_d, wins, losses = float(D.mean()), int((D > 0).sum()), int((D < 0).sum())
seed_d = [float((df[(df.ho == "step2") & (df.seed == s)].set_index("nhom").acc0 - df[(df.ho == "tcn_v2") & (df.seed == s)].set_index("nhom").acc0).mean()) for s in SEEDS]

pc = {g: ({n: np.mean([per_clip[(g, n, s)][0] for s in SEEDS], 0) for n in FAMS}, per_clip[(g, "tcn_v2", SEEDS[0])][2]) for g in GROUPS}
rng = np.random.default_rng(0)
def boot_once():
    ds = []
    for g in rng.choice(GROUPS, len(GROUPS)):
        c, n = pc[str(g)]; i = rng.integers(0, len(n), len(n))
        ds.append((c["step2"][i].sum() - c["tcn_v2"][i].sum()) / max(n[i].sum(), 1) * 100)
    return np.mean(ds)
B = np.array([boot_once() for _ in range(2000)]); lo, hi = np.percentile(B, [2.5, 97.5])

DELTA, MIN_GROUPS = 1.0, 5
if mean_d >= DELTA and wins >= MIN_GROUPS: verdict = "step2"
elif mean_d <= -DELTA and losses >= MIN_GROUPS: verdict = "tcn_v2"
else: verdict = "hoa"
print(f"Chênh acc0 trung bình theo nhóm (step2 trừ tcn_v2): {mean_d:+.2f} điểm | step2 hơn {wins}/8 nhóm, kém {losses}/8 | bootstrap 95%: [{lo:+.2f}, {hi:+.2f}]")
print("Chênh theo từng seed (trung bình 8 nhóm):", [f"{x:+.2f}" for x in seed_d], "| chênh acc45 trung bình:", f"{tab['chênh acc45'].mean():+.2f}")
print("KẾT LUẬN THEO QUY TẮC:", {"step2": "step2 THẮNG (chạy cell 6)", "tcn_v2": "tcn_v2 THẮNG (giữ tcn_v2 s1, không chạy cell 6)", "hoa": "HÒA, giữ tcn_v2 s1 (không chạy cell 6)"}[verdict])
(LOGO_CK / "logo_verdict.json").write_text(json.dumps({"verdict": verdict, "mean_diff_acc0": mean_d, "groups_step2_better": wins, "groups_step2_worse": losses,
    "boot95": [float(lo), float(hi)], "seed_diffs": seed_d, "rule": f"step2 thắng nếu chênh >= {DELTA} điểm và hơn >= {MIN_GROUPS}/8 nhóm; tcn_v2 thắng chiều ngược lại; còn lại hòa, giữ tcn_v2"}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Đã ghi", LOGO_CK / "logo_verdict.json")

Accuracy theo khung (%), trung bình 4 seed, nhóm bị giữ ra (chênh = step2 trừ tcn_v2):


,số clip,tcn_v2 acc0,step2 acc0,tcn_v2 acc45,step2 acc45,chênh acc0,chênh acc45
B2,66,66.2,65.7,64.0,65.3,-0.4,1.3
H1,33,75.1,74.1,71.3,74.4,-1.0,3.1
H2,18,90.2,90.4,85.9,90.2,0.2,4.3
H3,30,79.2,82.0,77.0,80.3,2.9,3.3
L1,36,98.1,97.6,95.1,97.5,-0.5,2.4
L2,50,96.0,96.1,93.3,96.1,0.1,2.7
L4,6,100.0,100.0,98.4,100.0,0.0,1.6
L5,27,100.0,99.9,96.5,99.9,-0.0,3.4


Chênh acc0 trung bình theo nhóm (step2 trừ tcn_v2): +0.15 điểm | step2 hơn 3/8 nhóm, kém 4/8 | bootstrap 95%: [-1.15, +1.48]
Chênh theo từng seed (trung bình 8 nhóm): ['+1.01', '-0.13', '-0.33', '+0.04'] | chênh acc45 trung bình: +2.76
KẾT LUẬN THEO QUY TẮC: HÒA, giữ tcn_v2 s1 (không chạy cell 6)
Đã ghi C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\checkpoints_logo\logo_verdict.json


In [6]:
#cell 6: CHỈ chạy khi cell 5 kết luận step2 THẮNG. Chọn seed bằng val loss (từ checkpoints_task9_v2), ngưỡng bằng val theo ACC_TARGET, ghi file chọn TRƯỚC khi nhìn test, rồi chạy test ĐÚNG 1 LẦN
vd = json.loads((LOGO_CK / "logo_verdict.json").read_text(encoding="utf-8"))
if vd["verdict"] != "step2":
    print("Kết luận là", vd["verdict"], "=> không chạy test step2; giữ nguyên kết quả test của tcn_v2 s1.")
else:
    f = CK / "test_result_step2.json"
    assert not f.exists(), f"Test step2 đã chạy ({f}). Không chạy lại."
    vls = {s: torch.load(ck_path("step2", s), map_location=dev, weights_only=False)["val_loss"] for s in SEEDS}
    seed = min(vls, key=vls.get)
    net = TCN(546).to(dev); net.load_state_dict(torch.load(ck_path("step2", seed), map_location=dev, weights_only=False)["w"]); net.eval()
    feat = VARIANTS["step2"]["feat"]
    with torch.no_grad(): tv = thr_table(net(feat(Xv, Mv)).softmax(-1), yv, Mv)
    display(tv.round(1)); okt = tv[tv["acc khi tin cậy>=ngưỡng"] >= ACC_TARGET]
    THR2 = float(okt["ngưỡng"].iloc[0]) if len(okt) else float(tv["ngưỡng"].iloc[-1])
    (CK / "final_model_v2_step2.json").write_text(json.dumps({"model": f"step2 s{seed}", "val_loss_theo_seed": vls, "nguong": THR2, "chon_tren": "val (manifest_v2); quyết định đổi họ theo kiểm chứng chéo 30_LOGO_Compare"}, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Đã chọn step2 s{seed}, ngưỡng {THR2} (ghi final_model_v2_step2.json trước khi xem test)")
    out = {}
    for sub, idx in SUB.items():
        with torch.no_grad(): P = net(feat(Xte[idx], Mte[idx])).softmax(-1)
        r = metrics(P, yte[idx], Mte[idx]); tt = thr_table(P, yte[idx], Mte[idx]); row = tt[tt["ngưỡng"] == THR2].iloc[0]
        out[sub] = {"n_video": len(idx), **r, "nguong": THR2, "acc_khi_tin_cay": float(row["acc khi tin cậy>=ngưỡng"]), "do_phu": float(row["độ phủ"]), "pct_video_dung_du_tin_cay": float(row["% video đúng và đủ tin cậy"])}
    f.write_text(json.dumps({"model": f"step2 s{seed}", "ket_qua": out}, indent=2, ensure_ascii=False), encoding="utf-8")
    show = pd.DataFrame(out).T[["n_video", "acc_all", "tb", "v50", "v60", "v70", "acc_khi_tin_cay", "do_phu", "pct_video_dung_du_tin_cay"]]
    show.columns = ["số video", "acc mọi khung", "TB 5 ngưỡng", "% video @0.5", "@0.6", "@0.7", f"acc khi tin cậy>={THR2}", "độ phủ", f"% video đúng và đủ tin cậy @{THR2}"]
    print(f"KẾT QUẢ TEST step2 s{seed}; all = 39 clip, nguoi_khac = person_01 (L3), sang = Sang (B1, B3):"); display(show.round(1)); print("Đã lưu:", f)

Kết luận là hoa => không chạy test step2; giữ nguyên kết quả test của tcn_v2 s1.
